# Trabajo Práctico N°4: Simulación de Eventos Discretos
## Modelado y Análisis de Puestos de Carga para Vehículos Eléctricos

**Objetivo:** Desarrollar un modelo de simulación evento a evento para analizar el comportamiento dinámico de una estación de carga de vehículos eléctricos. El estudio abarca desde el procesamiento de datos históricos para la caracterización estocástica de las variables de entrada hasta la ejecución de la simulación y el análisis de métricas de rendimiento en un sistema de colas multiserver.

In [ ]:
from google.colab import drive
# Montaje de Google Drive para el acceso a los datasets históricos
drive.mount('/content/drive')

## 1. Configuración del Entorno e Importación de Librerías

Se procede con la preparación del entorno de ejecución, incluyendo la instalación de la librería `fitter` para el modelado estadístico y la importación de módulos para el procesamiento de datos, visualización y simulación estocástica.

In [ ]:
pip install fitter

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from fitter import Fitter
import math
import random
from typing import List

## 2. Carga de Datos Inicial

En esta sección se carga el dataset para la generación de las Funciones de Densidad de Probabilidad (FDP)

In [ ]:
# Ruta de acceso a los datos
PATH_DATOS = "/content/drive/MyDrive/Colab Notebooks/TP 4 Simu/Datos/"

try:
    # Carga de datasets para el análisis de variables estocásticas
    df = pd.read_csv(PATH_DATOS + "EVChargingStationUsage.csv")

    print("Dataset inicializado correctamente.")
except FileNotFoundError as e:
    print(f"Error en la carga de recursos: {e}")

## 3. Obtención de la FDP del Tiempo de Carga (TC)

El Tiempo de Carga representa la duración de la sesión de carga. Se utiliza la columna `Charging Time (hh:mm:ss)` del dataset mapeada a minutos para determinar la distribución estocástica que mejor represente este proceso, y poder generar valores aleatorios para dicha distribución.

In [ ]:
def mostrar_histograma(data, title, xlabel):
    plt.figure(figsize=(7,5))
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Densidad de Probabilidad")
    plt.hist(data, bins=100, density=True)
    plt.show()

df["Charging Time (minutes)"] = (
    pd.to_timedelta(df["Charging Time (hh:mm:ss)"].dropna())
    .dt.total_seconds() / 60
)

mostrar_histograma(df["Charging Time (minutes)"], "Histograma del Tiempo de Carga", "Tiempo de Carga (minutos)")

Dado que la distribución tiene una cola derecha larga, se verá las distribuciónes que mejor se ajusten usando como criterio el `ks_statistic`.

In [ ]:
def mostrar_mejores_dist(data, nombre, criterio):
    """Ajusta y visualiza las mejores distribuciones para una serie de datos."""
    print(f"--- Ajustando distribuciones para {nombre} según {criterio} ---")
    f = Fitter(data)
    f.fit()
    display(f.summary(method=criterio))
    return f

f_tc = mostrar_mejores_dist(df["Charging Time (minutes)"], "Tiempo de Carga", "ks_statistic")

Se selecciona la distribución `Gumbel R`, pues tiene el ks_statistic más bajo, y también gana en sumsquare_error y kl_div.

In [ ]:
def mostrar_fdp(title, xlabel, f, fdp, data):
    plt.figure(figsize=(7,5))
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Densidad de Probabilidad")
    plt.hist(data, bins=100, density=True, alpha=0.5)
    f.plot_pdf(names=[fdp])
    plt.show()

def obtener_dist(f, funcion, nombre, data):
    """Extrae una distribución scipy lista para usar a partir de un Fitter ajustado."""
    if funcion not in f.fitted_param:
        disponibles = list(f.fitted_param.keys())
        raise ValueError(f"'{funcion}' no fue ajustada. Disponibles: {disponibles}")

    params_tuple = f.fitted_param[funcion]
    dist_class   = getattr(stats, funcion)
    distribucion = dist_class(*params_tuple)

    param_names  = (dist_class.shapes.split(", ") if dist_class.shapes else []) + ["loc", "scale"]
    params_dict  = dict(zip(param_names, params_tuple))

    print(f"La función elegida para {nombre} es: {funcion}")
    print(f"Con los parámetros: {params_dict}")
    print(f"Acotada entre: {data.min()} y {data.max()}")

    mostrar_fdp(f"FDP para {nombre}", f"{nombre} (minutos)", f, funcion, data)

    return distribucion

dist_tc = obtener_dist(f_tc, "gumbel_r", "Tiempo de Carga", df["Charging Time (minutes)"])

def rvs_truncado(dist, data):
    """
    Genera muestras de una distribución truncada por los límites originales
    de los datos, sin acumular probabilidad en los bordes.
    """
    low  = data.min()
    high = data.max()
    p_low  = dist.cdf(low)    # probabilidad acumulada hasta el límite inferior
    p_high = dist.cdf(high)   # probabilidad acumulada hasta el límite superior

    # Muestreamos uniforme SOLO en la franja de probabilidad válida
    u = np.random.uniform(p_low, p_high)

    # Transformada inversa: convierte probabilidades → valores
    return dist.ppf(u)

def generar_tc():
    """Generador estocástico de Tiempo de Carga para el motor de simulación."""
    return float(rvs_truncado(dist_tc, df["Charging Time (minutes)"]))

## 4. Obtención de la FDP del Intervalo de Arribos (IA)

El Intervalo de Arribos representa el tiempo entre que llega un vehículo al sistema y llega otro. Se utiliza la columna `Start Date` del dataset para calcular el tiempo entre llegadas, determinar la distribución estocástica que mejor represente este proceso, y poder generar valores aleatorios para dicha distribución.

In [ ]:
df["Start Date"] = pd.to_datetime(df["Start Date"].dropna(), format="%m/%d/%Y %H:%M")
df = df.sort_values("Start Date")
df["Interarrival Time"] = df["Start Date"].diff().dt.total_seconds() / 60
df["Interarrival Time"] = df["Interarrival Time"].replace(0, 0.5)

data_arribos = df["Interarrival Time"].dropna()

mostrar_histograma(data_arribos, "Histograma del Intervalo de Arribos", "Intervalo de Arribos (minutos)")

Dado que esta distribución también tiene una cola derecha muy larga, se verá las distribuciónes que mejor se ajusten usando como criterio el `ks_statistic`.

In [ ]:
f_ia = mostrar_mejores_dist(data_arribos, "Intervalo de Arribos", "ks_statistic")

Se selecciona la distribución `Landau`, pues por más que no sea la de menor ks_statistic, no está lejos, y tiene menor sumsquare_error y menor kl_div, los cuales también son importantes de analizar para Simulación.

In [ ]:
dist_ia = obtener_dist(f_ia, "landau", "Intervalo de Arribos", data_arribos)

def generar_ia():
    """Generador estocástico de Intervalo de Arribos para el motor de simulación."""
    return float(rvs_truncado(dist_ia, data_arribos))

## 5. Obtención del Costo Energético (CE) en función del TC

El Costo Energético representa el dinero gastado en la energía de los cargadores. Este depende de:
- El Tiempo de Carga (TC)
- El Precio de la Energía (PE): Valor constante que representa cuanto vale el kWh en CABA en Pesos

Para calcularlo, usaremos regresión lineal para obtener una función que nos permita pasar de TC a CE.


In [ ]:
PE = 108.48 # Precio Promedio de Energia en CABA, en Pesos

tiempo = df["Charging Time (minutes)"].values
energia = df['Energy (kWh)'].dropna().values

slope, intercept, r, p, se = stats.linregress(
    np.sort(tiempo),
    np.sort(energia)
)

print(f"Energía [kWh] = {slope:.4f} x Tiempo [min] + {intercept:.4f}  (R²={r**2:.4f})")
print(f"Costo [ARS] = Energía [kWH] x {PE} [ARS/kWH]")

def tiempo_a_costo_lineal(t):
    return float((slope * t + intercept) * PE)

## 6. Implementación del Motor de Simulación

Se define un modelo de simulación de eventos discretos para un sistema multi-servidor (N y N) con lógica de abandono (arrepentimiento). El sistema gestiona una lista de eventos futuros basada en arribos y salidas por cada puesto.

In [ ]:
# Variables de Control
N: int                        # Cantidad de puestos

#  Estado del reloj y eventos
T: float                      # Reloj actual
TF: float                     # Horizonte de simulación
TPLL: float                   # Próxima llegada global
TPS: List[float]              # Próxima salida por puesto

#  Estado del sistema
NS: List[int]                 # Autos en cada puesto (cola + servicio)

#  Contadores y acumuladores
CLL: int                      # Llegadas efectivas
CARR: int                     # Arrepentimientos
SPS: float                    # suma ponderada de personas en sistema
STC: float                    # Suma de tiempos de carga

ITO: List[float]              # Instante último cambio a ocioso por puesto
STO: List[float]              # Tiempo ocioso acumulado por puesto
SCE: List[float]              # Costo Energético acumulado por puesto

#  Resultados
PPS: float                    # Promedio de personas en sistema
PEC: float                    # Promedio en cola
PARR: float                   # % arrepentimiento
PTO: List[float]              # Proporción de ocio por puesto
CCE: List[float]              # Costo de Consumo Energetico por puesto

# Factor de comportamiento: probabilidad de que un usuario abandone la cola
PORCENTAJE_ARREPENTIMIENTO = 0.93

def condiciones_iniciales(cant_puestos: int):
    global TF, N
    global T, TPLL, TPS
    global NS
    global CLL, CARR, SPS, STC, ITO, STO, SCE

    # Variables de Control
    N = int(cant_puestos)

    # Reloj/eventos
    T = 0.0
    TF = 1000000.0
    TPLL = 0.0

    # Acumuladores
    CLL = 0
    CARR = 0
    SPS = 0.0
    STC = 0.0

    # Vectores
    TPS = [float("inf")] * N
    NS = [0] * N
    ITO = [0.0] * N
    STO = [0.0] * N
    SCE = [0.0] * N

def arrepentimiento(index_puesto) -> bool:
    global CARR, NS, PORCENTAJE_ARREPENTIMIENTO
    cant_autos = NS[index_puesto]

    if cant_autos <= 1:
        return False
    elif cant_autos <= 2:
        r = random.random()
        if r >= PORCENTAJE_ARREPENTIMIENTO:
            return False
        else:
            CARR += 1
            return True
    else:
        CARR += 1
        return True

def llegada():
    global TPLL, SPS, T, NS, CLL, STC, STO, ITO, TPS, N

    SPS += (TPLL - T) * (sum(NS))

    T = TPLL #Avanzamos en el tiempo

    IA = generar_ia() #Generamos el tiempo hasta la proxima llegada
    TPLL = T + IA #Calculamos el tiempo de llegada del proximo auto

    index_puesto = eleccion_server() #Subrutina
    if arrepentimiento(index_puesto):
        return

    # Llegada efectiva
    NS[index_puesto] += 1
    CLL += 1

    if NS[index_puesto] == 1:
        TC = generar_tc()
        STC += TC
        SCE[index_puesto] += tiempo_a_costo_lineal(TC)
        TPS[index_puesto] = T + TC
        STO[index_puesto] += (T - ITO[index_puesto])



def salida(index_puesto):
    global SPS, T, NS, STC, ITO, STO, TPS

    SPS += (TPS[index_puesto] - T) * (sum(NS))

    T = TPS[index_puesto]  # Avanzamos en el tiempo

    NS[index_puesto] -= 1  # Actualizo vector estado

    if NS[index_puesto] > 0:  # Evento Futuro Condicionado
        TC = generar_tc()
        STC += TC
        SCE[index_puesto] += tiempo_a_costo_lineal(TC)
        TPS[index_puesto] = T + TC  # Actualizo vector de tiempos de salida
    else:
        ITO[index_puesto] = T
        TPS[index_puesto] = float("inf")  # Actualizo vector de tiempos de salida

def eleccion_server():
    global N
    return random.randrange(N) # Distribución uniforme de la demanda

def menor_tps_index():
    global TPS
    return min(range(len(TPS)), key=TPS.__getitem__)

def calculo_resultados():
    global SPS, STC, SCE, CLL, CARR, STO, T, N
    total_arribos = CLL + CARR

    PPS = SPS / CLL if CLL > 0 else 0.0
    PEC = (SPS - STC) / CLL if CLL > 0 else 0.0
    PARR = CARR * 100 / total_arribos if total_arribos > 0 else 0.0

    PTO = []
    CCE = []

    for i in range(N):
        PTO.append(STO[i] * 100 / T if T > 0 else 0.0)

    for i in range(N):
        CCE.append(SCE[i] * 100 / CLL if CLL > 0 else 0.0)

    print(f"PPS = {round(PPS,2)}")
    print(f"PEC = {round(PEC,2)}")
    print(f"PTO = {[round(x, 2) for x in PTO]}")
    print(f"CCE = {[round(x, 2) for x in CCE]}")
    print(f"PARR = {round(PARR,2)}")


def simular(cant_puestos: int):
    global TPLL, TPS, T, TF, NS

    condiciones_iniciales(cant_puestos)

    while True:
        index_server = menor_tps_index()

        if TPLL <= TPS[index_server]:
            llegada()
        else:
            salida(index_server)

        if T <= TF:
            continue
        else: # Vaciamiento
            if any(x > 0 for x in NS):
                TPLL = float("inf")
                continue
            break
    calculo_resultados()

## 7. Experimentación y Análisis de Sensibilidad

Se ejecutan múltiples escenarios variando el número de puestos de carga (N) para identificar el punto de equilibrio operativo donde se minimiza el arrepentimiento y el tiempo de espera en cola, manteniendo una utilización eficiente del cargador.

In [ ]:
# Simulación para configuraciones de 1 a 40 puestos
for i in range(1, 41):
    print(f"\n--- ESCENARIO: N = {i} puestos ---")
    simular(cant_puestos=i)